# 0.0 Initial Library Import

In [ ]:
# Standard libraries
import sys, math, itertools, warnings, importlib, textwrap, random, ast, re, gc, pickle, json, os, sklearn, xgboost, joblib
import ast
import importlib
import itertools
import json
import math
import os
import pickle
import random
import re
import sys
import textwrap
import warnings
from datetime import date, datetime
from pathlib import Path
from typing import List

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
NOTEBOOK_UTILS_SRC = PROJECT_ROOT / "notebook_utils" / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

if str(NOTEBOOK_UTILS_SRC) not in sys.path:
    sys.path.append(str(NOTEBOOK_UTILS_SRC))

from paths import DATA_INTERMEDIATE, DATA_RAW, FIGURES, MODELS, PROJECT_ROOT, TABLES

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyfolio as pf
import seaborn as sns
from IPython.display import HTML, display
from scipy.optimize import minimize
from scipy.stats import entropy, norm, t

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
# Panda display options
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None) # Show all content of each column
pd.set_option('display.width', 1000)        # Set the display width to 1000 characters
pd.options.display.float_format = '{:,.5f}'.format
np.set_printoptions(precision=5, suppress=True)

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

# 0.1 Data import

In [ ]:
# Model setup, repo-relative paths, and light notebook guardrails
model = "regime_model"
model_round = "round_1"

model_input_path = DATA_RAW / "xls" / "input" / model
model_round_input_path = model_input_path / model_round
model_intermediate_path = DATA_INTERMEDIATE / model / model_round
model_table_output_path = TABLES / model / model_round
model_csv_output_path = model_table_output_path / "csv"
model_xls_output_path = model_table_output_path / "xls"
model_figure_output_path = FIGURES / model / model_round
model_ml_output_path = MODELS / model / model_round
research_liquidity_path = DATA_RAW / "research" / "liquidity" / "xls"
research_fear_greed_path = DATA_RAW / "research" / "fear_greed" / "csv"

for directory in [
    model_intermediate_path,
    model_table_output_path,
    model_csv_output_path,
    model_xls_output_path,
    model_figure_output_path,
    model_ml_output_path,
]:
    directory.mkdir(parents=True, exist_ok=True)


def assert_file_exists(path: Path, label: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    return path


def ensure_columns(df: pd.DataFrame, required_columns, df_name: str) -> None:
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise KeyError(f"{df_name} is missing required columns: {missing_columns}")


def ensure_non_empty(df: pd.DataFrame, df_name: str) -> None:
    if df.empty:
        raise ValueError(f"{df_name} is empty. Check the upstream parquet export.")

In [ ]:
import notebook_utils._formatting_functions as _formatting_functions
importlib.reload(_formatting_functions)
from notebook_utils._formatting_functions import nan_inf_summary

import notebook_utils._optimization_functions_37_v2 as _optimization_functions_37_v2
importlib.reload(_optimization_functions_37_v2)
from notebook_utils._optimization_functions_37_v2 import (
    optimization_ranges, 
    process_optmz_minmax, 
    optmz_loop_wrap, 
    process_optmz_breaks, 
    optmz_search_with_exclusions, 
    sum_combinations, 
    create_combinations, 
    text_to_dict, 
    outlier_bound, 
    optmz_loop_wrap_with_exclusions
    )


In [ ]:
# Main dictionary for time-serie data aggregation
sum_cols = ['mtm_pl', 'entry_pl', 'matched_shares', 'entry_side', 'entry_fees', 'exit_fees', 'exit_shares', 'pl_g', 'pl_n', 'fees']
sum_dict = {key: 'sum' for key in (sum_cols)}

In [ ]:

names = [
    # Data with XGB prob
    "partition_ins_80_001",
    "partition_ins_20_001",
    "partition_oos_001",
]

# Option A: load into a dict
loaded = {
    name: pd.read_parquet(model_intermediate_path / f"{name}.parquet", engine="pyarrow")
    for name in names
}
partition_ins_80_001 = loaded["partition_ins_80_001"]
partition_ins_20_001 = loaded["partition_ins_20_001"]
partition_oos_001    = loaded["partition_oos_001"]

print({k: v.shape for k, v in loaded.items()})


In [ ]:
partition_ins_80_001.head(2)

# 1.0 Non-ML Optimization

### - Importing list of variables to optimize

In [ ]:

GROUP_ALWAYS = [
    "atr_250",
    "atr_ema_a",
    "atr_ema_b",
    "atr_ktg",
    "atr_sma_a",
    "atr_sma_b",
    "beta",
    "cmf",
    "corr_1y",
    "corr_20d",
    "d_atr",
    "d_avol5",
    "d_avol50",
    "d_natr",
    "d_natr_ktg",
    "d_rsi",
    "kalmar_q",
    "pct_chg_open",
    "rsi",
    "rvol",
    "vol_20d",
    "vol_5d",
    "vol_60d",
    "spy_atr",
    "spy_rvol",
    # PCA / macro regime variables
    "PCA_Index_ma5",
    "PCA_ScaledIndex_ma5",
    "PCA_Index_ma20",
    "PCA_ScaledIndex_ma20",
    "PCA_Index_ma50",
    "PCA_ScaledIndex_ma50",
    "PCA_Raw_full",
    "PCA_Index_full",
    "fear_greed",
]

GROUP_CALENDAR = [
    "entry_hr_dec",
    "entry_hr_dec_to_close",
    "week_day_sin",
    "week_day_cos",
    "month_sin",
    "month_cos",
    "year_day_sin",
    "year_day_cos",
]

EXCLUDE_COLUMNS = {
    "mtm_pl",
    "pl_g",
    "pl_n",
    "fees",
    "Capital",
    "wins",
    "target_return",
    "target_down",
    "source_file",
    "entry_time",
    "exit_time",
    "normed_date",
    "symbol",
}

FEATURE_RULES = {
    "distance": lambda c: c.startswith("dist_"),
    "percent": lambda c: c.startswith("pct_"),
    "return": lambda c: c.startswith("ret_"),
    "dummy": lambda c: c.startswith("dumm_"),
    "volume_ratio": lambda c: ("vol" in c and "_rat" in c),
}

MANUAL_INCLUDE = []
MANUAL_EXCLUDE = []

def build_feature_list_from_columns(df):
    cols = df.columns.tolist()
    selected = []

    for col in GROUP_ALWAYS:
        if col in cols:
            selected.append(col)
    
    for col in GROUP_CALENDAR:
        if col in cols:
            selected.append(col)

    for _, rule in FEATURE_RULES.items():
        selected.extend([col for col in cols if rule(col)])

    selected.extend([col for col in MANUAL_INCLUDE if col in cols])

    selected = list(dict.fromkeys(selected))
    selected = [col for col in selected if col not in EXCLUDE_COLUMNS]
    selected = [col for col in selected if col not in MANUAL_EXCLUDE]

    return selected


In [ ]:
# Note: "_001" suffix means insample ("ins") train ("80") data with ML probability (w. ml_proba_1)
feature_columns = build_feature_list_from_columns(partition_ins_80_001)
if not feature_columns:
    raise ValueError("No feature columns were selected from partition_ins_80.")

missing_feature_columns = [col for col in feature_columns if col not in partition_ins_20_001.columns or col not in partition_oos_001.columns]
if missing_feature_columns:
    raise KeyError(f"Selected features missing from downstream partitions: {missing_feature_columns}")

print(f"variables:{len(feature_columns)}")
print(textwrap.fill(", ".join(feature_columns), width=250))

In [ ]:
probs = ['ml_proba_1']
optmz_list_all_plus = feature_columns + probs

print(f'Predicted values added: {len(optmz_list_all_plus)}')

### - Defining optimization ranges and steps

In [ ]:
drop_manual = []
all_ranges, var_stats = optimization_ranges(partition_ins_80_001, optmz_list_all_plus, drop_manual, 20)
all_ranges.drop(all_ranges[all_ranges['steps'] == 0].index, inplace=True)

all_ranges.tail(5)

### - Data Export

In [ ]:
# Data exports for review

outname = f"optmz ranges {datetime.now().strftime('%Y%m%d')}.xlsx"
all_ranges.to_excel(model_xls_output_path / outname, index = True, engine='openpyxl')


## 1.1 Optimization of single variables for break discovery

In [ ]:
# Create the dictionary with the index as the key and ['min', 'max'] columns as the values
target_vars = {index: (row[0.1], row[0.9], row['steps']) for index, row in all_ranges.iterrows()}
print(f'Target variables: {len(target_vars)}')

# Pass target dictionary to criteria dictionary that will be used by search algorithm
crit_lst = [{key: (j, '>=')} for key, (element1, element2, element3) in target_vars.items() for j in np.arange(element1, element2, element3)]
print(f'Search criteria: {len(crit_lst)}')

### - Single variables, positive (>=)

In [ ]:
## TESTING ALL INDIVIDUAL TARGETS
# Note: max capital can be found as the max (Capital) column
MAX_CAPITAL = 68500

crit_lst = [{key: (j, '>=')} for key, (element1, element2, element3) in target_vars.items() for j in np.arange(element1, element2, element3)]
single_pos0, optmz, optmz_bydate = optmz_loop_wrap(partition_ins_80_001, crit_lst, MAX_CAPITAL)


# Post-processing consolidates results (selecting first and rd from bottom as a low bar)
# 'extrm' stands for extreme values (first and nth from last: 2 rows per variable)
# 'all' stands for all values (20 rows per variable)
single_pos_extrm, single_pos_all = process_optmz_minmax(single_pos0, 'Sharpe ratio', -3)
gc.collect()

print(f'63 variables x 20 steps: {len(single_pos_all)}')
print(f'63 variables x 2 extreme values: {len(single_pos_extrm)}')

In [ ]:
outname = f"optimization all result - pos {datetime.now().strftime('%Y%m%d')} v1.xlsx"
single_pos_all.to_excel(model_xls_output_path / outname, index = True, engine='openpyxl')

### - Lib import

In [ ]:
import notebook_utils._fast_optimization_v1 as _fast_optimization_v1 
importlib.reload(_fast_optimization_v1)
from notebook_utils._fast_optimization_v1 import greedy_threshold_search

import notebook_utils._bayesian_optimization_v1 as _bayesian_optimization_v1
importlib.reload(_bayesian_optimization_v1)
from notebook_utils._bayesian_optimization_v1 import bayesian_greedy_threshold_search

import notebook_utils._dashboard_functions_one_symbol_v2 as _dashboard_functions_one_symbol_v2
importlib.reload(_dashboard_functions_one_symbol_v2)
from notebook_utils._dashboard_functions_one_symbol_v2 import dashboard

import notebook_utils._data_explore_functions as _data_explore_functions
importlib.reload(_data_explore_functions)
from notebook_utils._data_explore_functions import (
    cross_tabs, 
    explore_cross, 
    decile_summary, 
    line_chart_grid, 
    create_distance, 
    append_summary, 
    get_summary, 
    reset_summary, 
    count_outliers_by_std
    )

## 1.2 Fast Optimization

### - With all features

In [ ]:
df = partition_ins_80_001

res = greedy_threshold_search(df, optmz_list_all_plus, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("trials_per_feature:", res.logs["trials_per_feature"])


### - Excluding ML-probability

In [ ]:
res = greedy_threshold_search(df, feature_columns, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)

print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("trials_per_feature:", res.logs["trials_per_feature"])


### - By Branches (ex-ML probability): A, B, C and D

In [ ]:
# (a) Selected, (b) distances, (c) percents, (d) returns 
feats_a = GROUP_ALWAYS
feats_b = [b for b in feature_columns if b.startswith("dist_")]
feats_c = [c for c in feature_columns if c.startswith("pct_")]
feats_d = [d for d in feature_columns if d.startswith("ret_")]
        

In [ ]:
# (a) Selected
res = greedy_threshold_search(df, feats_a, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")

# (b) distances
res = greedy_threshold_search(df, feats_b, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")

# (c) percents
res = greedy_threshold_search(df, feats_c, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")
 
# (d) returns 
res = greedy_threshold_search(df, feats_d, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")


### - By Branches (w. ML probability): A, B, C and D

In [ ]:
# (a) Selected, (b) distances, (c) percents, (d) returns 
feats_a_ml = GROUP_ALWAYS + probs
feats_b_ml = [b for b in feature_columns if b.startswith("dist_")] + probs
feats_c_ml = [c for c in feature_columns if c.startswith("pct_")] + probs
feats_d_ml = [d for d in feature_columns if d.startswith("ret_")] + probs
        

In [ ]:
# (a) Selected
res = greedy_threshold_search(df, feats_a_ml, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")

# (b) distances
res = greedy_threshold_search(df, feats_b_ml, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")

# (c) percents
res = greedy_threshold_search(df, feats_c_ml, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")
 
# (d) returns 
res = greedy_threshold_search(df, feats_d_ml, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("")


### - Bayesian, by Branches (ex-ML probability): A, B, C and D

In [ ]:
res = bayesian_greedy_threshold_search(df=df, features=feats_a, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])
print("")

res = bayesian_greedy_threshold_search(df=df, features=feats_b, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])
print("")

res = bayesian_greedy_threshold_search(df=df, features=feats_c, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])
print("")

res = bayesian_greedy_threshold_search(df=df, features=feats_d, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])

# for step in res.logs["bo_trace"]:
#     print(step)

### - Bayesian, by Branches (w. ML probability): A, B, C and D

In [ ]:
res = bayesian_greedy_threshold_search(df=df, features=feats_a_ml, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])
print("")

res = bayesian_greedy_threshold_search(df=df, features=feats_b_ml, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])
print("")

res = bayesian_greedy_threshold_search(df=df, features=feats_c_ml, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])
print("")

res = bayesian_greedy_threshold_search(df=df, features=feats_d_ml, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])

# for step in res.logs["bo_trace"]:
#     print(step)

## 1.3 Optimization - INS (80/20) and OOS

In [ ]:
# Notes: optimization thresholds not sucessful as only the ins_80 has positive Sharpe

LONG_SHORT = -1
ENTRY_FEE =  3.5

crit_lst = [    
         {'ml_proba_1': (0.449, '>='), 'vol_5d': (0.1043, '<='), 'spy_rvol': (1.0461, '<='), 'fear_greed': (46.0, '>=')}
               ]

# INS-80%
ins_80, optmz_ins_80, optmz_bydate_ins_80 = optmz_loop_wrap(partition_ins_80_001, crit_lst, MAX_CAPITAL)
strat_ins_001, ins_bydate_001 = dashboard(optmz_ins_80, ENTRY_FEE, '80% Optmz', LONG_SHORT, 'short', 'max')
print(ins_80.head(5))
print("")

# INS-20%
ins_20, optmz_ins_20, optmz_bydate_ins_20 = optmz_loop_wrap(partition_ins_20_001, crit_lst, MAX_CAPITAL)
strat_ins_002, ins_bydate_002 = dashboard(optmz_ins_20, ENTRY_FEE, '20% Optmz', LONG_SHORT, 'short', 'max')
print(ins_20.head(5))
print("")

# OOS
oos, optmz_oos, optmz_bydate_oos = optmz_loop_wrap(partition_oos_001, crit_lst, MAX_CAPITAL)
strat_oos_001, oos_bydate_001 = dashboard(optmz_oos, ENTRY_FEE, 'OOS Optmz', LONG_SHORT, 'short', 'max')
print(oos.head(20))
print("")


In [ ]:
# Notes: optimization thresholds not sucessful as equity curves need to be monotonically increasing

# Grid Equity Curves
fig, axs = plt.subplots(1, 3, figsize=(20, 5))  # Create a 2x3 grid of subplots
datasets = [ins_bydate_001, ins_bydate_002, oos_bydate_001]
titles = ['G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20% Optmz', 'G/N Equity Curve: OOS Optmz']

# Plot each dataset on a separate subplot
for i, (data, title) in enumerate(zip(datasets, titles)):
    # ax = axs[i // 3, i % 3]
    ax = axs[i]
    line_chart_grid(ax, data, 'normed_date', 'cum_pl_g', 'cum_pl_n', '$ eqt', title)

# Hide the unused subplot if there is any
if len(datasets) < axs.size:
    for j in range(len(datasets), axs.size):
        fig.delaxes(axs.flatten()[j])
plt.tight_layout()  # Adjust spacing between subplots
plt.show()


## 1.4 Alt Optimization - INS (80/20) and OOS

In [ ]:
# Notes: optimization thresholds not sucessful as only the ins_80 has positive Sharpe

crit_lst = [    
        {'ml_proba_1': (0.449, '>='), 'ret_px_1m_ago': (-0.0196, '<='), 'ret_px_prev3': (-0.0011, '<=')}
               ]

# INS-80%
ins_80, optmz_ins_80, optmz_bydate_ins_80 = optmz_loop_wrap(partition_ins_80_001, crit_lst, MAX_CAPITAL)
strat_ins_001, ins_bydate_001 = dashboard(optmz_ins_80, ENTRY_FEE, '80% Optmz', LONG_SHORT, 'short', 'max')
print(ins_80.head(5))
print("")

# INS-20%
ins_20, optmz_ins_20, optmz_bydate_ins_20 = optmz_loop_wrap(partition_ins_20_001, crit_lst, MAX_CAPITAL)
strat_ins_002, ins_bydate_002 = dashboard(optmz_ins_20, ENTRY_FEE, '20% Optmz', LONG_SHORT, 'short', 'max')
print(ins_20.head(5))
print("")

# OOS
oos, optmz_oos, optmz_bydate_oos = optmz_loop_wrap(partition_oos_001, crit_lst, MAX_CAPITAL)
strat_oos_001, oos_bydate_001 = dashboard(optmz_oos, ENTRY_FEE, 'OOS Optmz', LONG_SHORT, 'short', 'max')
print(oos.head(20))
print("")


In [ ]:
# Notes: optimization thresholds not sucessful as equity curves need to be monotonically increasing

# Grid Equity Curves
fig, axs = plt.subplots(1, 3, figsize=(20, 5))  # Create a 2x3 grid of subplots
datasets = [ins_bydate_001, ins_bydate_002, oos_bydate_001]
titles = ['G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20% Optmz', 'G/N Equity Curve: OOS Optmz']

# Plot each dataset on a separate subplot
for i, (data, title) in enumerate(zip(datasets, titles)):
    # ax = axs[i // 3, i % 3]
    ax = axs[i]
    line_chart_grid(ax, data, 'normed_date', 'cum_pl_g', 'cum_pl_n', '$ eqt', title)

# Hide the unused subplot if there is any
if len(datasets) < axs.size:
    for j in range(len(datasets), axs.size):
        fig.delaxes(axs.flatten()[j])
plt.tight_layout()  # Adjust spacing between subplots
plt.show()
